# HW — Topic 12 · MLE로 모수 추정하기

**확률통계 · Topic 12** | 배점 25점 | 개인 과제

---

## ✍️ 제출자 정보 — 먼저 채우세요

| | |
|---|---|
| **학번** | (여기에 작성) |

> 파일명을 **`HW_학번_T12.ipynb`** 로 바꿔서 제출한다. (예: `HW_202512345_T12.ipynb`)
> ⚠️ **제출 방법과 기한은 PLATO · Google Classroom 공지**를 확인한다.

### 할 일

| 문제 | 내용 | 배점 |
|:-:|---|:-:|
| 1 | Exponential의 MLE 유도와 검증 | 10 |
| 2 | 손으로 못 푸는 MLE — Beta 분포 | 8 |
| 3 | 손실함수의 정체 | 7 |

⚠️ **제출 전 `런타임 → 모두 실행`** 으로 출력을 남길 것. 출력이 없으면 −2점.
문제 설명 전문과 채점 루브릭은 [T12_hw.md](T12_hw.md) 를 참고한다.

> 💡 **`scipy.optimize` 사용법** (자세한 예제는 Lab 12 「먼저 도구부터」 셀)
> - 모수 1개 → `optimize.minimize_scalar(f, bounds=(a, b), method="bounded")`
> - 모수 여러 개 → `optimize.minimize(f, x0=[...], method="Nelder-Mead")` — 함수는 모수를 **배열 하나**로 받는다
> - 결과 `res.x` 가 최소점이다. 넘기는 함수는 **NLL (Negative Log-Likelihood, 음의 로그 가능도)**

## Part 0. 준비

이 셀을 먼저 실행한다. **시드는 바꾸지 말 것.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize

rng = np.random.default_rng(20260302)   # ⚠️ 이 줄은 바꾸지 마세요
print("준비 완료")

## 문제 1 — Exponential의 MLE 유도와 검증 (10점)

$X_1, \ldots, X_n \sim \text{Exp}(\lambda)$, PDF는 $f(x) = \lambda e^{-\lambda x}$ $(x \ge 0)$ 이다.

### (a) $\ell(\lambda)$ 를 쓰고 미분해서 $\hat\lambda_{\text{MLE}}$ 를 유도한다 — 2점

이 셀을 더블클릭해 과정을 적는다. 수식은 `$...$` 로 쓴다.

> **답:** (여기에 작성)
>
> $\ell(\lambda) = $ …
>
> $\dfrac{d\ell}{d\lambda} = $ … $= 0$ $\;\Rightarrow\; \hat\lambda = $ …

### ✏️ TODO 1 — (b) 데이터 500개로 (a)의 공식을 계산한다 · 2점

`rng.exponential` 의 인자는 $\lambda$ 가 아니라 **평균 $1/\lambda$ (scale)** 다.

In [ ]:
TRUE_LAM = 0.4
x1 = rng.exponential(scale=1 / TRUE_LAM, size=500)

# TODO 1: (a)에서 유도한 공식으로 lambda_hat 을 계산하세요
lam_formula = 0.0
print(f"표본평균 = {x1.mean():.4f}")
print(f"공식으로 구한 lambda_hat = {lam_formula:.4f}   (참값 {TRUE_LAM})")

### ✏️ TODO 2 — (c-1) 격자 탐색으로 구한다 · 1점

$\ell(\lambda) = n\log\lambda - \lambda\sum x_i$ 를 격자 위에서 계산해 **가장 큰 곳**을 고른다.

In [ ]:
lam_grid = np.linspace(0.01, 2.0, 4000)
# TODO 2: 각 격자점의 log-likelihood 를 계산하고, 가장 큰 곳의 lambda 를 고르세요
#         힌트: len(x1) * np.log(lam_grid) - lam_grid * x1.sum()  /  np.argmax
loglik = np.zeros_like(lam_grid)
lam_by_grid = lam_grid[np.argmax(loglik)]
print(f"격자 탐색 lambda_hat = {lam_by_grid:.4f}")

### ✏️ TODO 3 — (c-2) `scipy.optimize` 로 구하고 세 값을 비교한다 · 2점

NLL 을 만들어 `minimize_scalar` 에 넘긴다. 모수가 1개다.

In [ ]:
def nll_exp(lam):
    # TODO 3: 음의 log-likelihood 를 돌려주세요.  힌트: -(n*log(lam) - lam*sum(x))
    return 0.0


res1 = optimize.minimize_scalar(nll_exp, bounds=(0.001, 5.0), method="bounded")
lam_by_opt = res1.x

print(f"공식     {lam_formula:.4f}")
print(f"격자     {lam_by_grid:.4f}")
print(f"최적화   {lam_by_opt:.4f}")

### ✏️ TODO 4 — (d) 일치성 확인 표 · 3점

$n$ = 10, 100, 1000, 10000 에서 각각 **200번** 데이터를 새로 만들어 $\hat\lambda$ 를 구하고,
그 **평균과 표준편차**를 표로 정리한다.

In [ ]:
REPS = 200
print(f"{'n':>7}{'mean of lam_hat':>18}{'sd of lam_hat':>16}")
for n in [10, 100, 1000, 10000]:
    # TODO 4: 크기 n 의 데이터를 REPS 번 만들어 각각 lambda_hat 을 구한 배열 est 를 만드세요
    #         힌트: [1 / rng.exponential(scale=1 / TRUE_LAM, size=n).mean() for _ in range(REPS)]
    est = np.zeros(REPS)
    print(f"{n:>7}{est.mean():>18.4f}{est.std():>16.4f}")

### (d) 서술 — 일치성이 보이는가? $n = 10$ 의 평균은 참값 0.4 와 비교해 어떤가? — 위 표에 포함된 점수

> **답:** (여기에 작성)

## 문제 2 — 손으로 못 푸는 MLE: Beta 분포 (8점)

### (a) 데이터를 만든다

In [ ]:
data = rng.beta(2, 5, size=300)     # 참값 alpha = 2, beta = 5
print(f"표본 {len(data)}개 · 평균 {data.mean():.4f}")

### ✏️ TODO 5 — (b) NLL 을 최소화해 $(\hat\alpha, \hat\beta)$ 를 구한다 · 3점

Beta 는 **연속분포**라 `stats.beta.logpdf(data, a, b)` 를 쓴다. 모수가 2개라 `minimize` 를 쓰고,
함수는 모수를 **배열 하나** `p` 로 받아 `a, b = p` 로 푼다.

In [ ]:
def nll_beta(p):
    a, b = p
    if a <= 0 or b <= 0:             # 모수는 양수여야 한다
        return np.inf
    # TODO 5: 음의 log-likelihood 를 돌려주세요.  힌트: -np.sum(stats.beta.logpdf(data, a, b))
    return 0.0


res2 = optimize.minimize(nll_beta, x0=[1, 1], method="Nelder-Mead")
a_hat, b_hat = res2.x
print(f"수치 MLE  alpha_hat = {a_hat:.4f},  beta_hat = {b_hat:.4f}")

### ✏️ TODO 6 — (c) 추정한 PDF를 히스토그램에 겹쳐 그린다 · 3점

In [ ]:
xs = np.linspace(0.001, 0.999, 400)
# TODO 6: 추정한 모수로 Beta PDF 를 계산하세요.  힌트: stats.beta.pdf(xs, a_hat, b_hat)
pdf_hat = np.zeros_like(xs)

plt.figure(figsize=(7, 4))
plt.hist(data, bins=30, density=True, alpha=0.5, label="data")
plt.plot(xs, pdf_hat, lw=2.5, label=f"Beta({a_hat:.2f}, {b_hat:.2f}) by MLE")
plt.xlabel("x")
plt.ylabel("density")
plt.title("Beta MLE fit")
plt.legend()
plt.show()

### ✏️ TODO 7 — (d) `stats.beta.fit` 과 비교한다 · 2점

`floc=0, fscale=1` 은 위치·척도를 고정하고 $\alpha, \beta$ 만 추정하라는 뜻이다. SciPy 도 MLE 를 쓴다.

In [ ]:
# TODO 7: stats.beta.fit(data, floc=0, fscale=1) 로 alpha, beta 를 구하세요 (돌려주는 값은 4개)
a_fit, b_fit = 0.0, 0.0
print(f"수치 MLE        alpha = {a_hat:.4f},  beta = {b_hat:.4f}")
print(f"stats.beta.fit  alpha = {a_fit:.4f},  beta = {b_fit:.4f}")

## 문제 3 — 손실함수의 정체 (7점)

### (a) Gaussian 잡음 → 제곱오차 — 3점

$y_i = f(x_i) + \varepsilon_i$, $\varepsilon_i \sim N(0, \sigma^2)$ 일 때 NLL 을 최소화하는 것이
$\sum (y_i - f(x_i))^2$ 을 최소화하는 것과 같음을 3~5줄로 보인다.

> **답:** (여기에 작성)

### (b) Laplace 잡음 → ? — 2점

Laplace PDF 는 $\frac{1}{2b}\exp\left(-\frac{|x - \mu|}{b}\right)$ 이다 (Topic 7).
오차를 이 분포로 가정하면 어떤 손실함수가 나오는가? 결과만 맞으면 된다.

> **답:** (여기에 작성)

### (c) (b)의 손실함수가 이상치에 더 강한 이유 — 2문장 · 2점

> **답:** (여기에 작성)

---

## ✅ 제출 전 점검

- [ ] 맨 위에 **학번**을 적었다
- [ ] `TODO 1~7` 을 모두 채웠다
- [ ] 문제 1(a) · 1(d) · 3(a)(b)(c) 의 **유도·서술을 작성했다**
- [ ] `런타임 → 모두 실행` 으로 **모든 출력이 남아 있다** (1(d)의 표, 2(c)의 그래프, 2(d)의 비교)
- [ ] 그래프에 **한글이 없다**
- [ ] 파일명을 **`HW_학번_T12.ipynb`** 로 바꿨다

**제출처와 기한, 채점 루브릭은 [T12_hw.md](T12_hw.md) · PLATO · Google Classroom 공지를 확인한다.**